# 全文检索

1. 分析器： 将文本拆分为关键词
2. 索引与召回： 将关键词存储到索引中，并在查询时进行召回

## 常见的全文检索技术

| 类别 | 代表方法 | 核心思路 |
|---|---|---|
| 传统统计方法 | **TF-IDF + 余弦相似度** | 根据词频和逆文档频率给词赋权，将查询与文档表示成稀疏向量，再计算相似度 |
| 传统统计方法 | **BM25** | 综合词频饱和、词的稀有程度和文档长度计算相关性 |
| 学习型方法 | **SPLADE** | 用神经网络学习词项权重，并扩展相关词项，生成稀疏向量 |
| 学习型方法 | **BGE-M3 的稀疏检索模式** | 用模型为文本中的 token 学习权重，再根据共同 token 的权重计算相关性 |
| 学习型方法 | **ELSER** | Elastic 的稀疏编码模型，将文本转换成带权重的词项，支持语义检索，官方主要推荐用于英文 |

## 词频统计

词频（TF，Term Frequency）表示一则文本中，每个词出现的频率

比如：`ERR4217：错误码ERR4217表示网络错误`

经过分析器后得到：`ERR4217、错误、码、ERR4217、表示、网络、错误`

然后经过词频统计得到：`ERR4217:2   错误:2   码:1   表示:1   网络:1`

BM25往往会进一步对词本身进行数字化：`15:2  7:2  11:1  5:1  3:1`

这样的数据结构可以看作是向量`[0,0,0,1,0,1,0,2,0,0,0,1,0,0,0,2]`的压缩格式

该向量大部分位置没有值，称之为**稀疏向量**

对应的，经过`embedding`模型得到的向量称之为**稠密向量**



我们需要在每个实体入库时加入它的词频统计

| id   | text  | dense     | sparse    |
| ---- | ----- | --------- | --------- |
| 1    | 文本1 | 稠密向量1 | 稀疏向量1 |
| 2    | 文本2 | 稠密向量2 | 稀疏向量2 |
| 3    | 文本3 | 稠密向量3 | 稀疏向量3 |


### 代码实现

In [ ]:
from pymilvus import AsyncMilvusClient, DataType
client = AsyncMilvusClient(
    uri="http://127.0.0.1:19530",
    db_name="deal_pilot",
)
collection_name = "analyzer_hybrid_demo"

In [ ]:
from pymilvus import Function, FunctionType

schema = client.create_schema(auto_id=False)
schema.add_field("id", DataType.INT64, is_primary=True)
# 待检索的文本字段需要开启分析器功能
schema.add_field(
    "text", 
    DataType.VARCHAR, 
    max_length=2048, 
    enable_analyzer=True,
    analyzer_params={"type": "chinese"}
)
schema.add_field("dense", DataType.FLOAT_VECTOR, dim=1024)

# 新增稀疏向量字段
schema.add_field(
    field_name="sparse",
    datatype=DataType.SPARSE_FLOAT_VECTOR,
)

# 新增稀疏向量入库函数
bm25_function = Function(
    name="text_to_bm25",
    input_field_names=["text"],
    output_field_names=["sparse"],
    function_type=FunctionType.BM25,
)
schema.add_function(bm25_function)

## 倒排索引

为了提升检索效率，我们可以使用倒排索引（Inverted Index）来存储词频统计信息。

倒排索引的结构类似于：

```
"ERR4217": 实体1:2
"错误": 实体1:2   实体2:1   实体3:1
"码": 实体1:1  实体3:1,
"表示": 实体1:1  实体2:1,
"网络": 实体1:1
```

### 代码实现

In [ ]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="dense",
    index_type="HNSW",
    metric_type="COSINE",
)

# 为稀疏向量添加倒排索引，并指定度量类型为BM25
index_params.add_index(
    field_name="sparse",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
)

await client.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params,
)

## 添加数据

In [ ]:

from deal_pilot.rag.embedding_model import create_embedding_model

documents = [
    {
        "id": 1,
        "text": "ERR4217：PKIX 路径构建错误；TLS_ALERT_CERTIFICATE_UNKNOWN；CA 中间证书轮换；请替换 intermediate.pem 并重启服务。"
    },
    {
        "id": 2,
        "text": "ERR4218：客户端请求没有得到响应。检查服务进程是否正常、入口防火墙是否允许流量，以及路由端口是否开放。"
    },
    {
        "id": 3,
        "text": "ERR5217：访问凭证已失效。重新签发 API token，再重新提交请求。"
    },
    {
        "id": 4,
        "text": "网关调用长时间没有返回：核对名称解析、出口路由和链路延迟。"
    },
    {
        "id": 5,
        "text": "远端实例无法到达：检查应用健康状态、网络 ACL 和负载均衡配置。"
    },
    {
        "id": 6,
        "text": "会话协商卡住时，查看 TLS 日志、证书链和中间代理转发规则。"
    },
    {
        "id": 7,
        "text": "TLS 握手失败，通常需要检查服务端证书是否过期、证书链是否完整。"
    },
    {
        "id": 8,
        "text": "PKIX 路径构建失败时，请验证根 CA 和中间 CA 是否都在信任列表中。"
    },
    {
        "id": 9,
        "text": "收到 unknown CA 告警后，确认客户端信任的证书颁发机构与服务端证书一致。"
    },
    {
        "id": 10,
        "text": "客户端连接超时可能由代理、DNS 解析或网络链路延迟造成。"
    },
]

embedding_model = create_embedding_model()
vectors = await embedding_model.aembed_documents([doc["text"] for doc in documents])
entities = [
    {**doc, "dense": vector}
    for doc, vector in zip(documents, vectors)
]
entities

In [7]:
await client.upsert(collection_name=collection_name, data=entities)

{'upsert_count': 10, 'cost': 0, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}

## BM25 检索

BM25 检索原理：

1. **分析问题**：将问题转换成词元
2. **查找候选实体**：查询倒排索引，找到包含这些词元的实体及其词频。
3. **计算相关性分数**：对每个候选实体，分别计算各查询词元的得分，再相加。每个词元的得分主要取决于：
   - **词频（TF）**：该词在当前实体文本中出现得越多，贡献通常越大，但增益逐渐减小，趋于饱和。
   - **逆文档频率（IDF）**：包含该词的实体越少，它的区分能力越强，权重越高；这里统计的是实体数量，不是词的总出现次数。
   - **文本长度**：结合当前文本长度与集合平均文本长度进行归一化，减轻长文本因词多而天然占优的问题。
4. **排序返回**：按总分从高到低排序，返回前 `limit` 个结果。分数越高，表示按 BM25 规则计算的词项相关性越高。

参考：[BM25 评分原理](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)。

### 代码实现

In [8]:
await client.load_collection(collection_name)

In [9]:
question = "ERR4217"
sparse_hits = await client.search(
    collection_name=collection_name,
    data=[question],  # BM25 路输入原始查询文本
    anns_field="sparse",
    limit=5,
    output_fields=["text"],
)

In [11]:
sparse_hits

data: [[{'id': 1, 'distance': 2.0070581436157227, 'entity': {'text': 'ERR4217：PKIX 路径构建错误；TLS_ALERT_CERTIFICATE_UNKNOWN；CA 中间证书轮换；请替换 intermediate.pem 并重启服务。'}}]]